# Data Science with Python: Pandas, Images, and a Cat vs. Dog CNN

This notebook is designed to run in **Google Colab**. It builds on the basic Python data structures and file-handling skills (lists, dicts, `with open(...)`) from the intro notebook, and applies them to a real data science workflow:

1. **Getting data into Colab** — downloading a dataset, uploading it to Google Drive, and unzipping it
2. **Opening and inspecting images** in Python
3. **Manipulating data with pandas**
4. **Processing image data** for machine learning
5. **Final Challenge:** train a basic Convolutional Neural Network (CNN) to classify images as **cat** or **dog**

Throughout the notebook, look for **🔨 Challenge** boxes — these are for you to complete yourself.

> **Note:** This notebook downloads its own data automatically (Section 1) using standard Python (`urllib`), so it works both in Google Colab and in a regular Jupyter installation with internet access. To open it in Colab: go to `colab.research.google.com`, choose "Upload", and select this `.ipynb` file.


## 1. Getting the Data into Google Colab

We need real cat and dog photos to work with. Instead of downloading a file to your computer and uploading it somewhere, we'll do everything with code: **one cell downloads the dataset directly from the internet straight into your Colab session.**

We'll use a dataset of 2,000 cat/dog photos that Google itself hosts for teaching purposes (it's the same dataset used in TensorFlow's official tutorials), so it's small, fast, and reliable.

### Step 1.1 -- Download the dataset

Just run the cell below. It tries a primary download link, and if that one is ever unreachable, it automatically falls back to a second mirror, so this should work every time without you needing to find or paste any link yourself.


In [ ]:
import os
import zipfile
import urllib.request

DATA_ROOT = "/content"
ZIP_PATH = os.path.join(DATA_ROOT, "cats_and_dogs_filtered.zip")
DATA_DIR = os.path.join(DATA_ROOT, "cats_and_dogs_filtered")

# Two mirrors of the same dataset, both hosted by Google.
# If the first one fails for any reason, we automatically try the second.
MIRROR_URLS = [
    "https://storage.googleapis.com/mledu-datasets/cats_and_dogs_filtered.zip",
    "https://download.mlcc.google.com/mledu-datasets/cats_and_dogs_filtered.zip",
]

def download_dataset():
    for url in MIRROR_URLS:
        try:
            print(f"Trying: {url}")
            urllib.request.urlretrieve(url, ZIP_PATH)
            if zipfile.is_zipfile(ZIP_PATH):
                print("Download succeeded and file looks valid.")
                return True
            print("  Downloaded file is not a valid zip, trying next mirror...")
        except Exception as e:
            print(f"  Failed ({e}), trying next mirror...")
    return False

already_have_valid_zip = os.path.exists(ZIP_PATH) and zipfile.is_zipfile(ZIP_PATH)

if not already_have_valid_zip:
    ok = download_dataset()
    if not ok:
        raise RuntimeError(
            "Could not download the dataset from any mirror. "
            "Check your internet connection and try running this cell again."
        )
else:
    print("Zip file already downloaded, skipping.")

size_mb = os.path.getsize(ZIP_PATH) / (1024 * 1024)
print(f"Zip file ready: {ZIP_PATH} ({size_mb:.1f} MB)")


### Step 1.2 -- Unzip the dataset

Now we extract the zip file's contents onto the Colab machine's disk, the same idea as Section 5 (Files) from the intro notebook, just applied to a zip full of images instead of a single text file.


In [ ]:
if not os.path.isdir(DATA_DIR):
    print("Extracting...")
    with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall(DATA_ROOT)
    print("Extraction complete!")
else:
    print("Already extracted, skipping.")

print("Top-level folders:", os.listdir(DATA_DIR))


### Step 1.3 -- Explore the folder structure

Before writing any data science code, always look at what you actually have. This dataset comes already split into a `train` folder and a `validation` folder, each containing a `cats` folder and a `dogs` folder.


In [ ]:
for root, dirs, files_in_dir in os.walk(DATA_DIR):
    level = root.replace(DATA_DIR, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in sorted(files_in_dir)[:3]:      # only show the first 3 files per folder
        print(f"{indent}  {f}")
    if len(files_in_dir) > 3:
        print(f"{indent}  ... and {len(files_in_dir) - 3} more files")


### Step 1.4 -- Define our cat and dog folders

For the rest of this notebook, we'll treat `train` and `validation` together as one big pool of images, and build our *own* train/test split later in Section 4, that's a more realistic workflow, and it means we only need to remember two lists of folders: one for cats, one for dogs.


In [ ]:
CATS_DIRS = [
    os.path.join(DATA_DIR, "train", "cats"),
    os.path.join(DATA_DIR, "validation", "cats"),
]
DOGS_DIRS = [
    os.path.join(DATA_DIR, "train", "dogs"),
    os.path.join(DATA_DIR, "validation", "dogs"),
]

for folder in CATS_DIRS + DOGS_DIRS:
    print(folder, "->", len(os.listdir(folder)), "images")


### Challenge 1

1. Using `os.listdir(...)` and `len(...)`, print the total number of cat images (adding up both folders in `CATS_DIRS`) and the total number of dog images.
2. Print whether the dataset is balanced (roughly the same number of cats and dogs) or not.
3. Pick any one folder from `CATS_DIRS` and print the names of its first 5 files (hint: `sorted(os.listdir(folder))[:5]`).


In [ ]:
# Your code here



## 2. Opening and Inspecting Images

We'll use the **Pillow** library (`PIL`), the standard way to open and manipulate images in Python, along with `matplotlib` to actually display them.


In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

# We pick the first cat image we find, instead of typing a filename by hand,
# since we don't know the exact filenames in advance.
sample_folder = CATS_DIRS[0]
sample_filename = sorted(os.listdir(sample_folder))[0]
image_path = os.path.join(sample_folder, sample_filename)
print("Opening:", image_path)

img = Image.open(image_path)

print("Format:", img.format)
print("Size (width, height):", img.size)
print("Mode:", img.mode)   # 'RGB' means 3 color channels

plt.imshow(img)
plt.axis("off")
plt.title("Example image")
plt.show()


### Images as arrays of numbers

A computer doesn't see a picture — it sees a grid of numbers. Converting an image to a NumPy array reveals this directly: a color image is a 3D array of shape `(height, width, 3)`, one value per color channel (Red, Green, Blue) for every pixel.


In [ ]:
import numpy as np

img_array = np.array(img)
print("Array shape:", img_array.shape)
print("Data type:", img_array.dtype)
print("Pixel value range:", img_array.min(), "to", img_array.max())

# The value of a single pixel at row 0, column 0
print("Top-left pixel (R, G, B):", img_array[0, 0])


### Resizing images

Real photos come in all shapes and sizes, but machine learning models need every input to be the exact same size. `PIL` makes resizing straightforward.


In [ ]:
img_resized = img.resize((128, 128))
print("New size:", img_resized.size)

plt.imshow(img_resized)
plt.axis("off")
plt.title("Resized to 128x128")
plt.show()


### 🔨 Challenge 2

1. Open **three** different images from your dataset (mix of cats and dogs).
2. For each image, print its original size and mode.
3. Resize each one to `64x64` and display all three side by side using `plt.subplot`.


In [ ]:
# Your code here



## 3. Organizing Data with Pandas

Instead of juggling separate lists of filenames and labels, we can build a **pandas DataFrame** — a table, much like a spreadsheet — where each row is one image and its metadata.

This connects directly back to the intro notebook's Section 6 (Data Processing): before, we built a list of dictionaries by hand from a CSV file. `pandas.DataFrame` does that same job, but with far more built-in tools for exploring, filtering, and summarizing data.


In [ ]:
import pandas as pd

VALID_EXTENSIONS = (".jpg", ".jpeg", ".png")
records = []

for folder in CATS_DIRS:
    for filename in os.listdir(folder):
        if filename.lower().endswith(VALID_EXTENSIONS):
            records.append({"filepath": os.path.join(folder, filename), "label": "cat"})

for folder in DOGS_DIRS:
    for filename in os.listdir(folder):
        if filename.lower().endswith(VALID_EXTENSIONS):
            records.append({"filepath": os.path.join(folder, filename), "label": "dog"})

df = pd.DataFrame(records)
df.head()


### Exploring the DataFrame

A few pandas methods you'll use constantly:


In [ ]:
print(df.shape)          # (number of rows, number of columns)
df.info()                 # column types and non-null counts


In [ ]:
df["label"].value_counts()   # how many images per class


### Filtering rows

Filtering a DataFrame works a lot like the list comprehensions with an `if` condition you used in the intro notebook, but on a whole table at once.


In [ ]:
cats_only = df[df["label"] == "cat"]
print("Number of cat images:", len(cats_only))
cats_only.head()


### Adding computed columns

Let's add a column with each image's file size in kilobytes, computed from the file itself — this uses `os.path.getsize`, another example of reading file metadata like we did with `os.listdir` earlier.


In [ ]:
df["size_kb"] = df["filepath"].apply(lambda path: os.path.getsize(path) / 1024)
df.head()


In [ ]:
df.groupby("label")["size_kb"].mean()


### 🔨 Challenge 3

1. Build a DataFrame like the one above from your own dataset.
2. Add a new column `"width"` and `"height"` by opening each image with `PIL` and reading `img.size`. (Tip: write a small function and use `.apply`, or loop through the DataFrame with `.iterrows()`.)
3. Use `.groupby("label")` to print the average width and height for cats vs. dogs.
4. Use `df[...]` filtering to find any images where the width is less than 50 pixels (these might be corrupt or too small to use).


In [ ]:
# Your code here



## 4. Processing the Images for a Machine Learning Model

To train a model, every image needs to become a same-sized numeric array, and every label needs to become a number (`0` for cat, `1` for dog, for example). This is the same "read → process" pipeline pattern from the intro notebook, just applied to images instead of text.


In [ ]:
IMG_SIZE = 64   # we'll resize every image to 64x64 pixels

def load_and_process_image(filepath, size=IMG_SIZE):
    img = Image.open(filepath).convert("RGB")     # make sure every image has 3 color channels
    img = img.resize((size, size))
    array = np.array(img) / 255.0                  # scale pixel values from [0, 255] to [0, 1]
    return array

# Try it on one image
sample_array = load_and_process_image(df.iloc[0]["filepath"])
print("Shape:", sample_array.shape)
print("Value range:", sample_array.min(), "-", sample_array.max())


### Building the full dataset

Now we apply this function to every row in the DataFrame to build our feature matrix `X` and label vector `y`.


In [ ]:
X = np.array([load_and_process_image(path) for path in df["filepath"]])
y = np.array([0 if label == "cat" else 1 for label in df["label"]])

print("X shape:", X.shape)   # (num_images, 64, 64, 3)
print("y shape:", y.shape)


### Splitting into train and test sets

We hold out part of the data to check how well the model generalizes to images it has never seen.


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training set:", X_train.shape, y_train.shape)
print("Test set:", X_test.shape, y_test.shape)


### 🔨 Challenge 4

1. Using the `df` you built in Challenge 3, apply `load_and_process_image` to every filepath to build `X` and `y`.
2. Split the data into training and test sets with an 80/20 split.
3. Print how many cats and dogs ended up in the training set (hint: `np.bincount(y_train)`).


In [ ]:
# Your code here



## 5. 🔨 Final Challenge: Cat vs. Dog Classifier with a Basic CNN

Now put everything together: file handling, pandas, image processing, and a bit of new material — a **Convolutional Neural Network (CNN)** — to classify images as cat or dog.

You don't need to design the CNN architecture from scratch. A simple starting architecture is given below using `tensorflow.keras`. Your job is to **fill in the pipeline around it**, reusing concepts from this notebook.

### What your solution must include

1. **Data loading** — reuse `CATS_DIRS` and `DOGS_DIRS` from Section 1 (the dataset is already downloaded and extracted).
2. **A pandas DataFrame** — with at least `filepath` and `label` columns, like Section 3. Print `df["label"].value_counts()` to confirm your classes are reasonably balanced.
3. **Image processing** — resize every image to a consistent size and normalize pixel values, like Section 4.
4. **Train/test split**.
5. **Model training** — using the CNN skeleton below (or your own).
6. **Evaluation** — report test accuracy, and show at least 5 test images together with the model's predicted label vs. the true label (use `plt.imshow` + `plt.title`, like Section 2).

### CNN skeleton


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

model = models.Sequential([
    layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),

    layers.Conv2D(32, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dense(1, activation="sigmoid"),   # 1 output: probability of "dog"
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()


### Training

Once your `X_train`, `y_train`, `X_test`, and `y_test` are ready from Section 4, training looks like this:


In [ ]:
history = model.fit(
    X_train, y_train,
    epochs=10,
    validation_data=(X_test, y_test)
)


### 🔨 Your tasks from here

1. **Plot the training history**: use `history.history["accuracy"]` and `history.history["val_accuracy"]` to plot training vs. validation accuracy over the epochs with `matplotlib`.
2. **Evaluate on the test set**:
   ```python
   test_loss, test_acc = model.evaluate(X_test, y_test)
   print("Test accuracy:", test_acc)
   ```
3. **Make predictions and visualize results**: pick 5 random images from `X_test`, run `model.predict(...)` on them, and display each image with its predicted label ("cat" or "dog") and its true label as the title. Use `plt.subplot` to show them in a single row, similar to Challenge 2.
4. **Reflect** (answer in a markdown cell below): 
   - Did your model overfit (training accuracy much higher than validation accuracy)? How can you tell from your plot?
   - Which pandas exploration steps from Section 3 could help you understand *why* the model might be making mistakes (e.g. image size, class balance)?
   - What is one thing you would change about your data pipeline if you had more time (more images, better resizing, data augmentation, etc.)?


In [ ]:
# Your code here: training history plot, evaluation, and prediction visualization



_Write your reflection answers here._


## Summary

| Step | Tool | Concept reused from earlier |
|---|---|---|
| Getting data into Colab | `urllib.request`, `zipfile` | Files (reading/writing on disk) |
| Exploring folders | `os.listdir`, `os.walk` | Loops, lists |
| Opening images | `PIL.Image.open` | Files |
| Images as numbers | `numpy` arrays | Data structures (nested sequences) |
| Organizing metadata | `pandas.DataFrame` | Dictionaries, CSV processing |
| Building the dataset | resizing + normalizing + `train_test_split` | Data processing pipeline (read → process → split) |
| Classification | `tensorflow.keras` CNN | Everything above, feeding into a model |

The pattern **collect → inspect → clean/transform → model → evaluate** is the core loop of almost every data science project, whether you're working with spreadsheets of numbers or thousands of images.

### Next steps
- Try **data augmentation** (`tf.keras.layers.RandomFlip`, `RandomRotation`) to artificially grow your training set.
- Try a larger or deeper CNN, or a pretrained model (transfer learning) with `tf.keras.applications`.
- Explore `pandas` further with real tabular datasets (not just image metadata) — e.g. `groupby`, `merge`, `pivot_table`.
